# CIS 731: Distributed Model Evaluation & MLflow Benchmarking

**Objective:** Evaluate Logistic Regression (with Feature Selection) versus XGBoost at scale on the IEEE-CIS Fraud dataset using local multi-node Spark execution (`joblib-spark`) and MLflow tracking.

## 1. Initialize Spark Session & MLflow Tracking

In [ ]:
import time
import mlflow
import joblib
from joblibspark import register_spark
from pyspark.sql import SparkSession

# Initialize Spark Session connected to local standalone cluster
spark = SparkSession.builder \
    .appName("IEEE-Fraud-CV-Benchmark-Notebook") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Register the Spark backend for scikit-learn joblib parallelism
register_spark()

# Connect to local MLflow tracking server
mlflow.set_tracking_uri("http://mlflow-server:5000")
mlflow.set_experiment("Fraud_Detection_CV_Benchmark")
print("Spark and MLflow successfully initialized.")

## 2. Load Data from Local Bronze Volume and Prepare Tensors

In [ ]:
BRONZE = "/opt/bitnami/spark/data/raw"

# Read training transactions
train_tx = spark.read.option("header", True).csv(f"{BRONZE}/train_transaction.csv")

# Bounded sample preparation for local cross-validation
pdf = train_tx.select("TransactionAmt", "card1", "C1", "D1", "isFraud").dropna().limit(100000).toPandas()
X = pdf.drop("isFraud", axis=1)
y = pdf["isFraud"].astype(int)

print(f"Loaded feature matrix shape: {X.shape}")

## 3. Define Scikit-Learn Pipelines & Distributed CV Benchmark Function

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

pipeline_lr = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif, k=3)),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_xgb = Pipeline([
    ('classifier', XGBClassifier(n_estimators=50, use_label_encoder=False, eval_metric='logloss'))
])

def benchmark_cv(pipeline, model_name, n_workers):
    mlflow.sklearn.autolog()
    with mlflow.start_run(run_name=f"{model_name}_{n_workers}_Workers"):
        start_time = time.time()
        
        # Distribute cross-validation folds across Spark worker executors
        with joblib.parallel_backend('spark', n_jobs=n_workers):
            cv_results = cross_validate(
                pipeline, X, y, 
                cv=5, 
                scoring=['f1_macro', 'roc_auc'], 
                n_jobs=n_workers
            )
            
        elapsed = time.time() - start_time
        mlflow.log_metric("cv_mean_f1_macro", cv_results['test_f1_macro'].mean())
        mlflow.log_metric("execution_time_seconds", elapsed)
        
        print(f"[{model_name} | Workers: {n_workers}] Completed in {elapsed:.2f}s | Mean F1: {cv_results['test_f1_macro'].mean():.4f}")
        return elapsed

## 4. Execute Benchmark & Measure Speedup

In [ ]:
t_lr_1 = benchmark_cv(pipeline_lr, "LogisticRegression", n_workers=1)
t_lr_2 = benchmark_cv(pipeline_lr, "LogisticRegression", n_workers=2)
print(f"\nLogistic Regression Speedup (1 vs 2 workers): {t_lr_1 / t_lr_2:.2f}x\n")

t_xgb_1 = benchmark_cv(pipeline_xgb, "XGBoost", n_workers=1)
t_xgb_2 = benchmark_cv(pipeline_xgb, "XGBoost", n_workers=2)
print(f"XGBoost Speedup (1 vs 2 workers): {t_xgb_1 / t_xgb_2:.2f}x")